# Ilerda: entrenar la NNUE (768 -> HIDDEN -> 1) amb bullet
Accelerator = GPU T4, Internet = On. Substitueix `USUARI` a la ruta de les dades (comprova-la amb la cel·la 2).

In [ ]:
!nvidia-smi
!curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -3

In [ ]:
!find /kaggle/input -name '*.txt' | head

In [ ]:
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!git clone --depth 1 https://github.com/jw1912/bullet /kaggle/working/bullet 2>&1 | tail -2
!cd /kaggle/working/bullet && cargo b -r --package bullet-utils 2>&1 | tail -3

In [ ]:
HIDDEN = 256  # 256 o 512
DATA = '/kaggle/input/datasets/srobis/ilerda-mix2-tb/mix2-tb.txt'
!/kaggle/working/bullet/target/release/bullet-utils convert --from text --input {DATA} --output /kaggle/working/data.bullet --threads 4
!/kaggle/working/bullet/target/release/bullet-utils shuffle --input /kaggle/working/data.bullet --mem-used-mb 8000 --output /kaggle/working/shuf.bullet

In [ ]:
p = '/kaggle/working/bullet/examples/simple.rs'
s = open(p).read()
s = s.replace('data/baseline.data', '/kaggle/working/shuf.bullet')
s = s.replace('"simple"', '"ilerda"')
s = s.replace('end_superbatch: 40', 'end_superbatch: 20')
s = s.replace('step: 18', 'step: 7')
s = s.replace('save_rate: 10', 'save_rate: 5')
s = s.replace('HIDDEN_SIZE: usize = 128', f'HIDDEN_SIZE: usize = {HIDDEN}')
s = s.replace('output_directory: "checkpoints"', 'output_directory: "/kaggle/working/checkpoints"')
open(p, 'w').write(s)

In [ ]:
!cd /kaggle/working/bullet && export PATH=$PATH:/root/.cargo/bin CUDA_PATH=/usr/local/cuda-13.0 RUSTFLAGS='-L /usr/local/nvidia/lib64' && cargo r -r --example simple --features cuda 2>&1 | tail -n 60

In [ ]:
OUT = f'/kaggle/working/ilerda-{HIDDEN}-mix2.bin'
!cp /kaggle/working/checkpoints/ilerda-20/quantised.bin {OUT}
!ls -l {OUT}